# Medication block EDA (Paulo)

Scope: the 23 drug columns (`metformin` → `metformin-pioglitazone`) + `readmitted`.

## Step 1 — Load & sanity check

Load with `na_values="?"` and `keep_default_na=False`, confirm the 101,766 × 50 shape, and check the value counts of the drug columns.

In [1]:
import pandas as pd

# keep_default_na=False so the text "None" (test not done) is not read as NaN
df = pd.read_csv("../data/raw/diabetic_data.csv", na_values="?", keep_default_na=False, low_memory=False)
assert df.shape == (101766, 50), df.shape
print("shape:", df.shape)
print("encounters:", df["encounter_id"].nunique(), "| patients:", df["patient_nbr"].nunique())

shape: (101766, 50)
encounters: 101766 | patients: 71518


In [2]:
drug_cols = df.loc[:, "metformin":"metformin-pioglitazone"].columns.tolist()
assert len(drug_cols) == 23
print("levels found in drug columns:", sorted(pd.unique(df[drug_cols].values.ravel())))
print("missing values in drug columns:", df[drug_cols].isna().sum().sum())

levels found in drug columns: ['Down', 'No', 'Steady', 'Up']
missing values in drug columns: 0


In [3]:
# Value counts per drug, one row per drug, ordered by number of encounters taking it
counts = (
    df[drug_cols].apply(lambda s: s.value_counts()).T
    .reindex(columns=["No", "Steady", "Up", "Down"]).fillna(0).astype(int)
)
counts["taking"] = counts[["Steady", "Up", "Down"]].sum(axis=1)
counts.sort_values("taking", ascending=False)

,No,Steady,Up,Down,taking
insulin,47383,30849,11316,12218,54383
metformin,81778,18346,1067,575,19988
glipizide,89080,11356,770,560,12686
glyburide,91116,9274,812,564,10650
pioglitazone,94438,6976,234,118,7328
rosiglitazone,95401,6100,178,87,6365
glimepiride,96575,4670,327,194,5191
repaglinide,100227,1384,110,45,1539
glyburide-metformin,101060,692,8,6,706
nateglinide,101063,668,24,11,703


In [4]:
df["readmitted"].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

### Step 1 — notes

- **Load is clean:** 101,766 encounters × 50 columns, 71,518 unique patients, as documented. The comma separator and `keep_default_na=False` are correct; `low_memory=False` removes the mixed-type warning on `payer_code`.
- **Drug columns are tidy:** all 23 use only `No` / `Steady` / `Up` / `Down` and have no missing values, so they need no cleaning.
- **Usage is very uneven:** insulin is used in 54,383 encounters and metformin in 19,988, while 9 drugs have fewer than 40 users and `examide` / `citoglipton` have none. Target: `<30` = 11,357 encounters (11.2%).

## Step 2 — Profile the medication block

`ydata-profiling` on the drug columns + `readmitted` only. Report saved to `reports/generated/`.

In [ ]:
from ydata_profiling import ProfileReport

In [ ]:
med = df[drug_cols + ["readmitted"]]

In [3]:
profile = ProfileReport(med, title="Medication block — diabetic_data", explorative=True)
profile.to_file("../reports/generated/profile-report-medications.html")

C:\Users\felic\OneDrive\Documentos\NOVA IMS\2 - Semester\Enterprise Data Science Bootcamp\Project\enterprise-data-science-bootcamp\.venv\Lib\site-packages\ydata_profiling\utils\dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/24 [00:00<?, ?it/s]

  8%|▊         | 2/24 [00:00<00:07,  2.82it/s]

 12%|█▎        | 3/24 [00:01<00:12,  1.64it/s]

 17%|█▋        | 4/24 [00:02<00:13,  1.49it/s]

100%|██████████| 24/24 [00:02<00:00,  9.75it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [4]:
# Alerts raised by the profile, grouped by type
alerts = pd.DataFrame(
    [(a.alert_type.name, a.column_name) for a in profile.get_description().alerts],
    columns=["alert", "column"],
)
for alert, cols in alerts.groupby("alert")["column"]:
    print(f"{alert} ({len(cols)}): {list(cols)}")

not_flagged = [c for c in drug_cols if c not in set(alerts["column"])]
print("Drug columns with no alert:", not_flagged)
print("Duplicate rows in this 24-column subset:", med.duplicated().sum())

CONSTANT (2): ['examide', 'citoglipton']
DUPLICATES (1): [None]
IMBALANCE (20): ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
Drug columns with no alert: ['insulin']
Duplicate rows in this 24-column subset: 99559


In [5]:
# Same picture as numbers: share of the most frequent level and number of encounters taking the drug
summary = pd.DataFrame({
    "n_levels": med[drug_cols].nunique(),
    "top_level": med[drug_cols].mode().iloc[0],
    "top_share_%": (med[drug_cols].apply(lambda s: s.value_counts(normalize=True).iloc[0]) * 100).round(2),
    "n_taking": (med[drug_cols] != "No").sum(),
    "n_missing": med[drug_cols].isna().sum(),
}).sort_values("n_taking")
summary

,n_levels,top_level,top_share_%,n_taking,n_missing
examide,1,No,100.00,0,0
citoglipton,1,No,100.00,0,0
glimepiride-pioglitazone,2,No,100.00,1,0
acetohexamide,2,No,100.00,1,0
metformin-pioglitazone,2,No,100.00,1,0
metformin-rosiglitazone,2,No,100.00,2,0
troglitazone,2,No,100.00,3,0
glipizide-metformin,2,No,99.99,13,0
tolbutamide,2,No,99.98,23,0
miglitol,4,No,99.96,38,0


In [6]:
med["readmitted"].value_counts(normalize=True).mul(100).round(2)

readmitted
NO     53.91
>30    34.93
<30    11.16
Name: proportion, dtype: float64

### Step 2 — notes

- **Zero variance:** `examide` and `citoglipton` are flagged CONSTANT (100% `No`). There is nothing to learn from them, so they are certain drops.
- **High imbalance:** 20 of the 23 drug columns are flagged IMBALANCE. Insulin is the only one with no alert (46.6% `No`). That flag alone isn't enough to drop a column, because metformin (~20k users) is flagged too. Step 3 needs a usage-count threshold. The 7 columns with 3 or fewer users are effectively empty.
- **No missing values** in any drug column. The DUPLICATES alert (99,559 rows) is expected: with only 24 categorical columns, most encounters share the same pattern, so it is not a data-quality issue. Target in this file: `<30` = 11.16%, `>30` = 34.93%, `NO` = 53.91%.